# 플래너 챗봇 실제 대화 테스트

이 노트북은 API 서버를 거치지 않고 `agents.todo_creation.planner.pipeline.run()`을 직접 호출합니다.

확인할 흐름:
- 플랜 생성과 무관한 대화는 `out_of_scope` 안내만 반환하고 플랜을 생성하지 않는지
- 목표 정보가 부족하면 헷갈리는 핵심만 꼬리 질문으로 묻는지
- 정보가 충분하면 TODO/캘린더 후보를 생성하는지

환경변수는 프로젝트 루트 `.env`를 읽습니다. `LLM_PROVIDER=runpod`이면 `RUNPOD_PLANNER_ENDPOINT_URL`을, 그 외에는 `QWEN_BASE_URL`/`QWEN_MODEL`을 사용합니다.

In [1]:
import os
import sys
from datetime import date, datetime
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

from adapters.todo_creation.qwen_llm import DEFAULT_QWEN_MODEL, QwenLLM
from adapters.todo_creation.runpod_llm import RunPodQwenLLM
from agents.todo_creation.planner.pipeline import PlannerPorts, get_debug_state, run
from agents.todo_creation.schemas import PlannerInput


def build_real_planner_ports() -> PlannerPorts:
    provider = (os.getenv("LLM_PROVIDER", "qwen") or "qwen").strip().lower()
    if provider == "runpod":
        endpoint_url = os.getenv("RUNPOD_PLANNER_ENDPOINT_URL", "").strip()
        if not endpoint_url:
            raise RuntimeError("LLM_PROVIDER=runpod 이면 RUNPOD_PLANNER_ENDPOINT_URL 이 필요합니다.")
        llm = RunPodQwenLLM(
            endpoint_url=endpoint_url,
            api_key=os.getenv("RUNPOD_API_KEY", "EMPTY") or "EMPTY",
            adapter="planner",
            model=os.getenv("QWEN_MODEL", DEFAULT_QWEN_MODEL) or DEFAULT_QWEN_MODEL,
            max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "2400")),
        )
        base_llm = RunPodQwenLLM(
            endpoint_url=endpoint_url,
            api_key=os.getenv("RUNPOD_API_KEY", "EMPTY") or "EMPTY",
            adapter="base",
            model=os.getenv("QWEN_MODEL", DEFAULT_QWEN_MODEL) or DEFAULT_QWEN_MODEL,
            max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "2400")),
        )
        return PlannerPorts(llm=llm, classifier=base_llm, validator=base_llm)

    base_url = os.getenv("QWEN_BASE_URL", "").strip()
    model = os.getenv("QWEN_MODEL", "").strip() or DEFAULT_QWEN_MODEL
    if not base_url:
        raise RuntimeError("QWEN_BASE_URL 이 필요합니다. 예: http://localhost:11434/v1")
    llm = QwenLLM(
        base_url=base_url,
        model=model,
        api_key=os.getenv("QWEN_API_KEY", "EMPTY") or "EMPTY",
        max_tokens=int(os.getenv("PLANNER_MAX_TOKENS", "2400")),
    )
    return PlannerPorts(llm=llm, classifier=llm, validator=llm)


ports = build_real_planner_ports()
thread_id = None
TODAY = date.today()
USER_ID = "notebook-user"

print("ROOT:", ROOT)
print("TODAY:", TODAY)
print("LLM:", type(ports.llm).__name__)
print("MODEL:", getattr(ports.llm, "model", None))

ROOT: /Users/areum/FINAL-PROJECT/mongle-ai
TODAY: 2026-06-25
LLM: RunPodQwenLLM
MODEL: Qwen/Qwen2.5-7B-Instruct


/Users/areum/FINAL-PROJECT/mongle-ai/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
def reset_chat() -> None:
    global thread_id
    thread_id = None
    print("thread reset")


def _dump_result(result):
    payload = result.model_dump(mode="json")
    print("kind:", payload.get("kind"))
    print("thread_id:", payload.get("thread_id"))
    if payload.get("kind") == "follow_up":
        print("question:", payload.get("question"))
        print("missing_aspects:", payload.get("missing_aspects"))
    elif payload.get("kind") == "out_of_scope":
        print("message:", payload.get("message"))
    else:
        print("summary_text:", payload.get("summary_text"))
        print("todos:")
        pprint(payload.get("todos"))
        print("calendar_events:")
        pprint(payload.get("calendar_events"))
    return payload


async def send(message: str):
    global thread_id
    print("USER:", message)
    result = await run(
        PlannerInput(
            user_id=USER_ID,
            message=message,
            today=TODAY,
            thread_id=thread_id,
        ),
        ports=ports,
        now=datetime.now(),
    )
    thread_id = result.thread_id
    return _dump_result(result)


def debug_state():
    if not thread_id:
        print("thread_id 없음")
        return None
    state = get_debug_state(thread_id=thread_id, ports=ports)
    pprint(state)
    return state

## 1. 무관 질문 차단

`kind`가 `out_of_scope`여야 하고, `todos`/`calendar_events`가 생성되면 안 됩니다.

In [ ]:
reset_chat()
await send("슈퍼스타 k 우승하고 싶은데 플랜을 어떻게 짜는게 좋을까?")

In [ ]:
await send("8월 8일에 경기가 예정되어 있어요")

In [ ]:
reset_chat()
await send("흑백요리사 우승하고 싶은데 어떻게 플랜을 짜는게 좋을까?")

## 2. 플랜 요청이지만 정보 부족

`kind`가 `follow_up`이어야 합니다. 부족한 정보 중 하나를 질문해야 정상입니다.

In [ ]:
reset_chat()
await send("정처기 공부 계획 짜줘")

## 3. 꼬리 질문에 이어서 답하기

위 셀에서 받은 질문에 맞춰 한 번씩 답해보세요. 같은 `thread_id`로 이어집니다.

예시 답변을 그대로 실행해도 되고, 문자열을 바꿔서 직접 대화해도 됩니다.

In [ ]:
await send("필기이고 시험은 3일 뒤야")

In [ ]:
await send("하루 2시간 가능하고 기출 1회독 했어. 비전공자야")

## 4. 한 번에 충분한 정보 제공

`kind`가 `candidates`이면 정상입니다. 오늘 날짜 task는 `todos`, 미래 날짜 task는 `calendar_events`에 들어갑니다.

In [ ]:
reset_chat()
await send("3일 뒤 정보처리기사 실기 시험이야. 하루 2시간 가능하고 SQL까지 봤고 전공자야. 남은 기간 공부 계획 짜줘")

## 5. 직접 대화용 셀

아래 문자열만 바꿔가며 실행하세요. 새 대화를 시작하려면 `reset_chat()`을 먼저 실행합니다.

In [ ]:
reset_chat()
await send("철인 삼종 경기에 출전하고 싶어요")

In [ ]:
debug_state()

## 6. 미지 목표·질문 횟수·말투 회귀 테스트

다음 시나리오에서 시험 질문이 나오지 않는지, 꼬리질문이 최대 두 번인지, 사용자 노출 문장에 `몽글`이 한 번만 포함되는지 확인합니다.

In [ ]:
reset_chat()
await send("흑백요리사 우승하고 싶어")
# 첫 질문에 답한 뒤 아래 두 줄을 순서대로 실행하세요.
# await send("아직 날짜는 정하지 않았고 가정 요리 경험은 있어요")
# await send("주 4회 가능하고 대표 메뉴를 완성하는 게 목표예요")
# debug_state()

In [ ]:
reset_chat()
chat_reply = await send("배고프다")
visible_text = chat_reply.get("message", "")
print("몽글 사용 횟수:", visible_text.count("몽글"))

---

In [ ]:
reset_chat()
await send("철인 삼종 경기에 출전하고 싶어")
# 추가 질문에 답해 플랜이 생성되면 첫 30일 상세 일정과 실제 목표일 일정이 함께 있는지 확인하세요.
# long_state = debug_state()
# print(long_state)

In [ ]:
await send("경기일은 9월30일이고 입문자입니다")

In [ ]:
await send("주 3회 가능합니다")

In [ ]:
long_state = debug_state()
print(long_state)

---

In [ ]:
reset_chat()
await send("웹 개발자로 취업 준비 계획을 어떻게 짜는게 좋을까?")
# 추가 질문에 답해 플랜이 생성되면 첫 30일 상세 일정과 실제 목표일 일정이 함께 있는지 확인하세요.
# long_state = debug_state()
# print(long_state)

In [ ]:
await send("올해 하반기 안에 웹 개발자 중소기업이라도 들어가고 싶어")

In [ ]:
await send("면접 연습이라던가 영어를 좀 잘해야되지않을까?")


---

In [3]:
reset_chat()
await send("수박 빨리 먹기 대회 출전하고 싶어요")

thread reset
USER: 수박 빨리 먹기 대회 출전하고 싶어요


qwen follow_up parse fail (attempt 1): non-JSON response: 경기일이 언제인가요? 그리고 현재 어떤 수준의 운동을 하고 계신가요?


kind: follow_up
thread_id: eaa52a8c-98b2-4cd4-9337-4def199008fb
question: 경기일이 언제인가요? 그리고 현재 어떤 수준의 운동을 하고 계신가요, 몽글?
missing_aspects: ['event_date', 'current_level', 'weekly_cadence']


{'kind': 'follow_up',
 'thread_id': 'eaa52a8c-98b2-4cd4-9337-4def199008fb',
 'question': '경기일이 언제인가요? 그리고 현재 어떤 수준의 운동을 하고 계신가요, 몽글?',
 'missing_aspects': ['event_date', 'current_level', 'weekly_cadence']}

In [4]:
await send("경기일은 8월 10일이고 하루에 수박 한 통씩은 꼭 먹는 것 같아")

USER: 경기일은 8월 10일이고 하루에 수박 한 통씩은 꼭 먹는 것 같아


qwen follow_up parse fail (attempt 1): non-JSON response: 경기일이 언제인가요? 그리고 현재의 운동 수준과 관련된 경험은 어떤가요?
qwen follow_up parse fail (attempt 1): non-JSON response: 하루에 몇 번 운동할 수 있는 시간이 있으신가요, 몽글?


kind: follow_up
thread_id: eaa52a8c-98b2-4cd4-9337-4def199008fb
question: 주 몇 회 훈련 가능한지, 몽글?
missing_aspects: ['current_level', 'weekly_cadence']


{'kind': 'follow_up',
 'thread_id': 'eaa52a8c-98b2-4cd4-9337-4def199008fb',
 'question': '주 몇 회 훈련 가능한지, 몽글?',
 'missing_aspects': ['current_level', 'weekly_cadence']}

In [5]:
await send("한 통 먹는데 15분정도 걸리는데 10분안팎으로 줄이고 싶어 매일 훈련 가능해")

USER: 한 통 먹는데 15분정도 걸리는데 10분안팎으로 줄이고 싶어 매일 훈련 가능해


qwen follow_up parse fail (attempt 1): non-JSON response: 하루에 몇 번 운동할 수 있는 시간이 있나요, 몽글?
qwen plan parse fail (attempt 1): missing required JSON keys: days
qwen plan parse fail (attempt 2): missing required JSON keys: days
planner output invalid; using deterministic fallback: missing required JSON keys: days


kind: candidates
thread_id: eaa52a8c-98b2-4cd4-9337-4def199008fb
summary_text: 처음 초안이 목표와 맞지 않아, 지금 확인된 정보만으로 기본 실행안을 다시 잡았어요. 확인되지 않은 정보는 current_level 정보는 확인되지 않아 일반적인 수준으로 가정, weekly_cadence 정보는 확인되지 않아 일반적인 수준으로 가정으로 잡았어요. 상세 일정은 2026-07-24까지 제공해요. 그 이후 2026-08-10까지는 목표에 맞춰 실행 범위를 넓히고 점검하며 마무리하는 흐름으로 이어가면 돼요, 몽글.
todos:
[{'due_date': '2026-06-25', 'tags': ['수박대회'], 'title': '현재 수준 기록'}]
calendar_events:
[{'due_date': '2026-07-01', 'tags': ['수박대회'], 'title': '기초 체력 30분'},
 {'due_date': '2026-07-07', 'tags': ['수박대회'], 'title': '기술 동작 20분'},
 {'due_date': '2026-07-12', 'tags': ['수박대회'], 'title': '주 2회 실행'},
 {'due_date': '2026-07-18', 'tags': ['수박대회'], 'title': '회복 상태 점검'},
 {'due_date': '2026-07-24', 'tags': ['수박대회'], 'title': '다음 단계 조정'}]


{'kind': 'candidates',
 'thread_id': 'eaa52a8c-98b2-4cd4-9337-4def199008fb',
 'todos': [{'title': '현재 수준 기록', 'due_date': '2026-06-25', 'tags': ['수박대회']}],
 'calendar_events': [{'title': '기초 체력 30분',
   'due_date': '2026-07-01',
   'tags': ['수박대회']},
  {'title': '기술 동작 20분', 'due_date': '2026-07-07', 'tags': ['수박대회']},
  {'title': '주 2회 실행', 'due_date': '2026-07-12', 'tags': ['수박대회']},
  {'title': '회복 상태 점검', 'due_date': '2026-07-18', 'tags': ['수박대회']},
  {'title': '다음 단계 조정', 'due_date': '2026-07-24', 'tags': ['수박대회']}],
 'summary_text': '처음 초안이 목표와 맞지 않아, 지금 확인된 정보만으로 기본 실행안을 다시 잡았어요. 확인되지 않은 정보는 current_level 정보는 확인되지 않아 일반적인 수준으로 가정, weekly_cadence 정보는 확인되지 않아 일반적인 수준으로 가정으로 잡았어요. 상세 일정은 2026-07-24까지 제공해요. 그 이후 2026-08-10까지는 목표에 맞춰 실행 범위를 넓히고 점검하며 마무리하는 흐름으로 이어가면 돼요, 몽글.',
 'personalization_patch': {'preferences': ['과일', '경기'],
  'constraints': ['정처기 없음', '출전 기간 1개월']}}